Setup

In [ ]:
import requests, json

BASE_URL = "http://127.0.0.1:8080/api"
PASSWORD = "Arix2026!"
token = None

def pretty(data):
    print(json.dumps(data, indent=2, ensure_ascii=False))

def call(method, path, **kwargs):
    headers = kwargs.pop("headers", {})
    if token:
        headers["Authorization"] = f"Bearer {token}"
    response = requests.request(method, f"{BASE_URL}{path}", headers=headers, **kwargs)
    print(f"{method} {path} -> {response.status_code}")
    try:
        return response.json()
    except ValueError:
        return response

Login 

In [ ]:
EMAIL = "ana.lopez@gmail.com"

data = call("POST", "/auth/login", json={"email": EMAIL, "password": PASSWORD})
token = data["data"]["access_token"]
me = data["data"]["user"]
pretty(me)

Registrar nuevo cliente

In [ ]:
data = call("POST", "/auth/register", json={
    "full_name": "Nombre Apellido",
    "email": "nuevo.cliente@gmail.com",
    "password": PASSWORD,
    "phone": "+504 9999-0000",
    "address": "Tegucigalpa"
})
token = data["data"]["access_token"]
pretty(data["data"]["user"])

Mi perfil

In [ ]:
data = call("GET", "/users/me")
pretty(data)

Listar tiendas

In [ ]:
data = call("GET", "/stores")
pretty(data)

Mi tienda (si eres Admin de Tienda)

In [ ]:
data = call("GET", "/store/me")
pretty(data)

Categorías

In [ ]:
data = call("GET", "/categories")
pretty(data)

Catálogo de productos

In [ ]:
data = call("GET", "/products")
pretty(data)

Detalle de un producto

In [ ]:
PRODUCT_ID = 1

data = call("GET", f"/products/{PRODUCT_ID}")
pretty(data)

Crear producto (Admin de Tienda)

In [ ]:
data = call("POST", "/store/products", json={
    "category_id": 1,
    "name": "Producto Nuevo",
    "slug": "producto-nuevo",
    "description": "Descripción del producto",
    "price": 499.00,
    "sku": "SKU-001",
    "initial_stock": 10,
    "min_stock": 3
})
pretty(data)

Inventario de un producto (Admin de Tienda)

In [ ]:
PRODUCT_ID = 1

data = call("GET", f"/store/products/{PRODUCT_ID}/inventory")
pretty(data)

Hacer una compra (Cliente)

In [ ]:
data = call("POST", "/orders/checkout", json={
    "items": [
        {"product_id": 1, "quantity": 1}
    ],
    "shipping_address": "Tegucigalpa, Honduras",
    "payment_method": "CREDIT_CARD",
    "card_last_digits": "1234"
})
pretty(data)

Mis órdenes (Cliente)

In [ ]:
data = call("GET", "/orders")
pretty(data)

Detalle de una orden

In [ ]:
ORDER_ID = 1

data = call("GET", f"/orders/{ORDER_ID}")
pretty(data)

Pedidos de mi tienda (Admin de Tienda)

In [ ]:
data = call("GET", "/store/orders")
pretty(data)

Cambiar estado de un pedido (Admin de Tienda)

In [ ]:
STORE_ORDER_ID = 1

data = call("PUT", f"/store/orders/{STORE_ORDER_ID}/status", json={"status": "PREPARING"})
pretty(data)

Generar/ver facturas de una orden (Cliente)

In [ ]:
ORDER_ID = 1

data = call("GET", f"/orders/{ORDER_ID}/invoices")
pretty(data)

Descargar PDF de una factura

In [ ]:
INVOICE_ID = 1

response = requests.get(f"{BASE_URL}/invoices/{INVOICE_ID}/download", headers={"Authorization": f"Bearer {token}"})
with open("factura.pdf", "wb") as f:
    f.write(response.content)
print("Guardado: factura.pdf")

Dejar una reseña (Cliente, debe haber comprado el producto)

In [ ]:
PRODUCT_ID = 1

data = call("POST", f"/products/{PRODUCT_ID}/reviews", json={
    "rating": 5,
    "comment": "Excelente producto"
})
pretty(data)

Ver reseñas de un producto

In [ ]:
PRODUCT_ID = 1

data = call("GET", f"/products/{PRODUCT_ID}/reviews")
pretty(data)

Agregar a favoritos (Cliente)

In [ ]:
PRODUCT_ID = 1

data = call("POST", "/favorites", json={"product_id": PRODUCT_ID})
pretty(data)

Ver mis favoritos

In [ ]:
data = call("GET", "/favorites")
pretty(data)

Crear ticket de soporte (Cliente)

In [ ]:
data = call("POST", "/tickets", json={
    "store_id": 1,
    "subject": "Consulta sobre mi pedido",
    "description": "Tengo una duda sobre el tiempo de entrega.",
    "priority": "MEDIUM"
})
pretty(data)

Mis tickets (Cliente)

In [ ]:
data = call("GET", "/tickets")
pretty(data)

Tickets de mi tienda (Admin de Tienda)

In [ ]:
data = call("GET", "/store/tickets")
pretty(data)

Responder un ticket

In [ ]:
TICKET_ID = 1

data = call("POST", f"/tickets/{TICKET_ID}/messages", json={"message": "Su respuesta aquí"})
pretty(data)

Ver un ticket completo

In [ ]:
TICKET_ID = 1

data = call("GET", f"/tickets/{TICKET_ID}")
pretty(data)

Iniciar conversación

In [ ]:
OTHER_USER_ID = 2
STORE_ID = 1

data = call("POST", "/chats", json={"other_user_id": OTHER_USER_ID, "store_id": STORE_ID})
pretty(data)

Enviar mensaje en un chat

In [ ]:
CHAT_ID = 1

data = call("POST", f"/chats/{CHAT_ID}/messages", json={"content": "Hola, tengo una pregunta"})
pretty(data)

Ver mis conversaciones

In [ ]:
data = call("GET", "/chats")
pretty(data)

Mis notificaciones

In [ ]:
data = call("GET", "/notifications")
pretty(data)

Listar todos los usuarios (Super Admin)

In [ ]:
data = call("GET", "/admin/users")
pretty(data)

Crear administrador de tienda (Super Admin)

In [ ]:
data = call("POST", "/admin/users/store-admins", json={
    "full_name": "Nuevo Admin",
    "email": "nuevo.admin@arix.com",
    "password": PASSWORD,
    "phone": "+504 9999-0001",
    "address": "Tegucigalpa"
})
pretty(data)

Crear tienda (Super Admin)

In [ ]:
data = call("POST", "/admin/stores", json={
    "business_name": "Nueva Tienda",
    "slug": "nueva-tienda",
    "description": "Descripción",
    "contact_email": "contacto@nuevatienda.com",
    "admin_user_id": None
})
pretty(data)

Listar todas las tiendas (Super Admin)

In [ ]:
data = call("GET", "/admin/stores")
pretty(data)

Suspender o reactivar una tienda (Super Admin)

In [ ]:
STORE_ID = 1

data = call("PUT", f"/admin/stores/{STORE_ID}/status", json={"status": "SUSPENDED"})
pretty(data)

Bloquear o reactivar un usuario (Super Admin)

In [ ]:
USER_ID = 4

data = call("PUT", f"/admin/users/{USER_ID}/status", json={"status": "BLOCKED"})
pretty(data)

Historial de auditoría (Super Admin)

In [ ]:
data = call("GET", "/admin/audit-logs")
pretty(data)